In [1]:
import shap

from tokenizers import ByteLevelBPETokenizer

from transformers import PreTrainedTokenizerFast

#from transformers_interpret import SequenceClassificationExplainer

import joblib

import pandas as pd

import torch

import inspect

import numpy as np

from collections import defaultdict

from sklearn.model_selection import StratifiedKFold, train_test_split

c:\Users\ddellamo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokens_len = 8192

test_size = 0.2

n_splits = 5

seed = 42

### Import of the tokenizer and the model to be explained

In [3]:
tokenizer_vocab_path  = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\vocab.json'
tokenizer_merges_path = r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\pretrained\DutchEHRTokenizer\merges.txt' 

# Inizializzazione del tokenizer che effettua una
# tokenizzazione basata su Byte Pair Encoding (BPE),
# algoritmo di tokenizzazione che unisce coppie di byte o 
# caratteri più frequenti in sequenze. In questo modo è 
# in grado di creare nuovi tokens basati sulla frequenza delle
# coppie di caratteri nei dati di addestramento. Questo approccio
# aiuta a gestire parole rare o sconosciute creando token basati
# su parti comuni di parole
tokenizer = ByteLevelBPETokenizer(
    vocab = tokenizer_vocab_path,
    merges = tokenizer_merges_path,
    add_prefix_space = True
)

tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer._tokenizer)
tokenizer.add_special_tokens({'pad_token': '<pad>'})
tokenizer.add_special_tokens({'mask_token': '<mask>'})

model_joblib = joblib.load(r'C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\ODIN\sandboxes\09_consults\results\HTransformer1D\2025_02_13_16_18_CLS\best_model.joblib')

model = model_joblib['model']

print(inspect.signature(model.forward))

def f(x):
    
    #print(list(x), type(x))
    
    tv = tokenizer(
        list(x), 
        padding="max_length", 
        max_length=tokens_len, 
        truncation=True, 
        return_tensors = 'pt'
    )

    #print("Sono passato qui")
    
    tokens = tv['input_ids'].cuda()

    masks = tv['attention_mask'].type(torch.bool).cuda()

    #print(tokens, tokens.shape)

    #print(tv)
    outputs = model(tokens, masks).detach().cpu().softmax(1).numpy()
    
    #print("Ho finito la funzione f")
    return outputs

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

### Import of the consults

In [ ]:
dataset_path = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet'

df = pd.read_parquet(dataset_path)

labels = df['label']
consults = df['consult']


indices = np.arange(consults.shape[0])

trainval_indices, test_indices = train_test_split(
    indices,
    test_size=test_size, 
    stratify=labels, 
    random_state=seed
)

consults_test = consults.iloc[test_indices]
labels_test = labels.iloc[test_indices]

consults_trainval = consults.iloc[trainval_indices]
labels_trainval = labels.iloc[trainval_indices]

In [4]:
skf = StratifiedKFold(
    n_splits = n_splits,
    shuffle=True,
    random_state=seed
)

In [7]:
explainer = shap.Explainer(
    model = f, 
    masker = tokenizer,
    output_names = ['No CVRM', 'Yes CVRM'],
    seed=seed
)

In [ ]:
shap_words = defaultdict(list)

for split, (train_indices, val_indices) in enumerate(skf.split(consults_trainval, labels_trainval)):
    
    consults_valtest = pd.concat([consults_trainval.iloc[val_indices], consults_test], axis=0)
    
    for i, consult in enumerate(consults_valtest):
        shap_values = explainer([consult])
        
        probas = f([consult])
        
        values = shap_values.values[0]
        base = shap_values.base_values[0]
        label = df.iloc[i]['label']

        for word in shap_values.data[0]:
            shap_words[word].append([values, base, probas, label])
    
    break

PartitionExplainer explainer: 2it [00:24, 24.96s/it]               
PartitionExplainer explainer: 2it [00:16, 16.80s/it]               
PartitionExplainer explainer: 2it [00:35, 35.79s/it]               
PartitionExplainer explainer: 2it [00:17, 17.22s/it]               
PartitionExplainer explainer: 2it [00:33, 33.96s/it]               
PartitionExplainer explainer: 2it [00:25, 25.82s/it]               
PartitionExplainer explainer: 2it [00:20, 20.94s/it]               
PartitionExplainer explainer: 2it [00:32, 32.11s/it]               
PartitionExplainer explainer: 2it [00:21, 21.54s/it]               
PartitionExplainer explainer: 2it [00:16, 16.85s/it]               
 94%|█████████▍| 470/498 [00:11<00:00, 28.07it/s] 

KeyboardInterrupt: 

### Prova

In [9]:
prova = [min(consults, key=len)]

prova = [consults[7]]

In [ ]:
explainer = shap.Explainer(
    model = f, 
    masker = tokenizer,
    output_names = ['No CVRM', 'Yes CVRM'],
    seed=seed
)

In [ ]:
shap_values = explainer(prova)

shap.plots.text(shap_values)

In [ ]:
from IPython.core.display import HTML

shap_values = explainer(prova)

HTML(shap.plots.text(shap_values, display=False))
with open('rendered.html', 'w') as file:
     file.write(shap.plots.text(shap_values, display=False))

PartitionExplainer explainer: 2it [00:16, 16.68s/it]               


---

In [21]:
shap_words['SpecialismeNaam']

[[array([[ 2.20151901e-06, -2.20147597e-06],
         [ 2.20151901e-06, -2.20147597e-06],
         [ 2.20151901e-06, -2.20147597e-06],
         ...,
         [-3.45128415e-05,  3.45127949e-05],
         [-3.45128415e-05,  3.45127949e-05],
         [-3.45128415e-05,  3.45127949e-05]]),
  array([0.99888808, 0.00111196]),
  array([[0.9917493 , 0.00825078]], dtype=float32),
  0]]